The purpose of this notebook is to go through the BMRB .str files and extract H-NMR information to a useable format.

**1. Create a csv file that matched BMRB id to corresponding str file**

In [6]:
# ---------------------------------------------------------------
# Create a CSV manifest of all BMRB NMR-STAR (.str) files
# Columns: bmrb_id, str_path
# Example output:
# bmse000001, /Users/.../bmse000001/bmse000001.str
# bmse000002, /Users/.../bmse000002/bmse000002.str
# ---------------------------------------------------------------

from pathlib import Path
import pandas as pd
import re

# --- Set this to your root directory containing all the bmse*/bmst* subfolders ---
DATA_ROOT = "bmrb_nmrstar/"

root = Path(DATA_ROOT).expanduser().resolve()
if not root.exists():
    raise SystemExit(f"Error: {root} does not exist")

# --- Find all .str files recursively ---
str_files = sorted(root.rglob("*.str"))
print(f"Found {len(str_files)} .str files under {root}")

# --- Extract the ID from filename or folder name ---
records = []
for fp in str_files:
    # Try to extract bmse/bmst ID from either file or parent folder name
    m = re.search(r'(bmse\d{6,}|bmst\d{6,})', str(fp), re.IGNORECASE)
    if m:
        bmrb_id = m.group(1).lower()
    else:
        bmrb_id = fp.stem.lower()
    records.append({"bmrb_id": bmrb_id, "str_path": str(fp)})

# --- Save to CSV ---
df = pd.DataFrame(records).drop_duplicates().sort_values("bmrb_id").reset_index(drop=True)

out_dir = root.parent / "data" / "processed"
out_dir.mkdir(parents=True, exist_ok=True)
out_csv = out_dir / "bmrb_file_manifest.csv"

df.to_csv(out_csv, index=False)
print(f"Manifest saved to: {out_csv}")
print(df.head(10))


Found 3629 .str files under /Users/alfred/Documents/GitHub/DDLS_project/code/get_bmrb_data/bmrb_nmrstar
Manifest saved to: /Users/alfred/Documents/GitHub/DDLS_project/code/get_bmrb_data/data/processed/bmrb_file_manifest.csv
      bmrb_id                                           str_path
0  bmse000001  /Users/alfred/Documents/GitHub/DDLS_project/co...
1  bmse000002  /Users/alfred/Documents/GitHub/DDLS_project/co...
2  bmse000003  /Users/alfred/Documents/GitHub/DDLS_project/co...
3  bmse000004  /Users/alfred/Documents/GitHub/DDLS_project/co...
4  bmse000005  /Users/alfred/Documents/GitHub/DDLS_project/co...
5  bmse000006  /Users/alfred/Documents/GitHub/DDLS_project/co...
6  bmse000007  /Users/alfred/Documents/GitHub/DDLS_project/co...
7  bmse000008  /Users/alfred/Documents/GitHub/DDLS_project/co...
8  bmse000010  /Users/alfred/Documents/GitHub/DDLS_project/co...
9  bmse000011  /Users/alfred/Documents/GitHub/DDLS_project/co...


**2. Create a simple NMR txt file for each entry**